### Model Parameter 설정

In [ ]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:5])

### 1) 모델 클래스 유형

#### OpenAI (텍스트 완성 모델 — 레거시)

| 구분 | `OpenAI` | `ChatOpenAI` |
|---|---|---|
| 엔드포인트 | `/v1/completions` (텍스트 완성) | `/v1/chat/completions` (채팅) |
| 입력 / 출력 | 문자열 → **문자열(`str`)** | 메시지 → **`AIMessage`** (`.content`로 본문 접근) |
| 사용 가능 모델 | 완성(completions) 지원 모델만 (예: `gpt-3.5-turbo-instruct`) | 현재 대부분의 OpenAI 모델 |

> `gpt-5-mini` 같은 채팅 모델을 `OpenAI` 클래스로 호출하면 404 오류가 납니다. 아래 `OpenAI` 실습은 모델 종료로 실행하지 않고 설명용으로만 남겨 두었습니다.

In [ ]:
# 계정의 모델 목록 조회 (OpenAI SDK 직접 사용)
# 주의: 이미 종료(deprecated)된 모델도 목록에 포함될 수 있으므로, 목록에 있다고 해서 호출이 된다는 보장은 없음
from openai import OpenAI as OpenAIClient

client = OpenAIClient(api_key=OPENAI_API_KEY)
ids = [m.id for m in client.models.list().data]
print("gpt-3.5-turbo-instruct" in ids)                    # 목록에 있는지
print([i for i in ids if "instruct" in i])                # instruct 계열 전체

In [ ]:
from langchain_openai import OpenAI

# [설명용 — 실행하지 않음]
# - OpenAI 클래스는 레거시 텍스트 완성(/v1/completions) 전용 클래스입니다.
# - model 을 지정하지 않으면 LangChain 기본값(gpt-3.5-turbo-instruct)이 사용됩니다.
# - 채팅 모델(예: gpt-5-mini)은 이 클래스로 호출할 수 없습니다. (404: This is a chat model ...)
# - 모델 목록에 보이더라도 종료된 모델(예: gpt-3.5-turbo-instruct-0914)은 호출되지 않습니다.

# llm = OpenAI(model="gpt-3.5-turbo-instruct-0914")
# print(llm.model_name)  # 사용 중인 모델명 확인

# result = llm.invoke("한국의 대표적인 관광지 3군데를 추천해 주세요.")
# print(type(result))    # str 반환 (ChatOpenAI 는 AIMessage)
# print(result)

# [실행 결과]
# NotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-3.5-turbo-instruct-0914` has been deprecated,
# learn more here: https://platform.openai.com/docs/deprecations', 'type': 'invalid_request_error',
# 'param': None, 'code': 'model_not_found'}}

#### ChatOpenAI

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")
print(model.model_name)

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "이 시스템은 여행 전문가입니다."),
    ("human", "{user_input}"),
])

chain = chat_prompt | model
result = chain.invoke({"user_input": "안녕하세요? 한국의 대표적인 관광지 3군데를 추천해 주세요."})

print(type(result))
print(result.content)

#### Groq

In [ ]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(GROQ_API_KEY[:5])

In [ ]:
# uv add groq

In [ ]:
from groq import Groq

llm = Groq(api_key=GROQ_API_KEY)
print(llm)  # 기본 모델 확인

try:
    # Groq API 호출
    chat_completion = llm.chat.completions.create(
        messages=[            
            {
                "role": "user",
                "content": "한국의 대표적인 관광지 3군데를 추천해 주세요.",
            }
        ],
        model="openai/gpt-oss-120b",
        temperature=0.7,
        max_tokens=300,
        top_p=1,
        stop=None,
        stream=False,
    )

    converted_message = chat_completion.choices[0].message.content
    print(converted_message)

except Exception as e:
    print(f"Error calling Groq API: {e}")
